In [8]:
# Install qiskit-machine-learning only if missing (avoid %pip getcwd errors)
try:
    import qiskit_machine_learning
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'qiskit-machine-learning'], cwd='/tmp')

# QSVC starter (based on Qiskit ML tutorial)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Robust imports for qiskit-machine-learning (handles different versions)
try:
    from qiskit_machine_learning.algorithms import QSVC
except Exception:
    try:
        from qiskit_machine_learning.algorithms.classifiers import QSVC
    except Exception:
        raise ImportError('Could not import QSVC from qiskit_machine_learning.algorithms')

# Prefer FidelityQuantumKernel on modern qiskit-machine-learning installs
KernelClass = None
try:
    from qiskit_machine_learning.kernels import QuantumKernel as KernelClass
except Exception:
    pass
if KernelClass is None:
    try:
        from qiskit_machine_learning.kernels import FidelityQuantumKernel as KernelClass
    except Exception:
        try:
            from qiskit_machine_learning.kernels import FidelityStatevectorKernel as KernelClass
        except Exception:
            try:
                from qiskit_machine_learning.kernels.quantum_kernel import QuantumKernel as KernelClass
            except Exception:
                raise ImportError('Could not find a compatible quantum kernel class. Check qiskit-machine-learning version.')

from qiskit.circuit.library import ZZFeatureMap

np.random.seed(42)
X = np.random.randn(120, 2)
y = (X[:,0]*X[:,1] > 0).astype(int)

fm = ZZFeatureMap(feature_dimension=2, reps=1)
qk = KernelClass(feature_map=fm)
clf = QSVC(quantum_kernel=qk)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)
print("Acc:", accuracy_score(y_te, pred))


Acc: 0.7333333333333333
